In [23]:
import os
import cv2
import PIL
import math
import glob
import fiona
import torch
import random
import tifffile
import rasterio
import numpy as np
from PIL import Image
from ultralytics import YOLO
from fiona.crs import from_epsg
import matplotlib.pyplot as plt
from rasterio.features import shapes
from rasterio.transform import from_origin
from shapely.geometry import shape, mapping

In [24]:
main_folder = "/home/jazzy/sem-farm/data" 
# main_tif_path = os.path.join(main_folder,"marhara_satellite_image.tif")

# main_tif_path = "D:/BISAG/rooftop drone/marhara_transparent_mosaic_group1_3857_modified.tif"

main_tif_path = "/home/jazzy/sem-farm/Spatial_temporal/data/2012.tif"
output_subregions_folder = os.path.join(main_folder,"cropped") # Cropped Output
subregion_size=256

input_dir_containing_jpg = output_subregions_folder # Cropped JPG images
output_dir_containing_jpg = os.path.join(main_folder,"predicted") # Output Binary predicted images
#model_path = "D:/BISAG/rooftop/best300_early_stop_270.pt" # path to the Model

merged_predicted_image_path= "/home/jazzy/sem-farm/Spatial_temporal/output/22out.jpg" # Merged Predicted Image Directory path

border_width = 4
    
output_folder = "/home/jazzy/sem-farm/Spatial_temporal/shapefile"
georeferenced_image_path = os.path.join(output_folder,"22spa.tif")
vector_shape_file_path = os.path.join(output_folder,"22spa.shp")

pixel_resolution=0.05 #In Meters

In [25]:
PIL.Image.MAX_IMAGE_PIXELS = None

In [26]:
os.makedirs(output_subregions_folder,exist_ok=True)
os.makedirs(output_dir_containing_jpg, exist_ok=True)

In [27]:
def get_georeferencing_info(tif_path):
    with rasterio.open(tif_path) as dataset:
        transform = dataset.transform
        crs = dataset.crs
    return transform, crs

# Function to georeference a JPG image
def geo_reference_jpg(jpg_path, output_tif_path, transform, crs):
    # Read the JPG image
    img = cv2.imread(jpg_path)

    # Get the dimensions of the image
    height, width, channels = img.shape

    # Create a new rasterio dataset
    new_transform = from_origin(transform.c, transform.f, transform.a, -transform.e)
    with rasterio.open(
        output_tif_path, 'w',
        driver='GTiff',
        height=height,
        width=width,
        count=channels,
        dtype=img.dtype,
        crs=crs,
        transform=new_transform,
    ) as dst:
        for i in range(1, channels + 1):
            dst.write(img[:, :, i - 1], i)

In [28]:
def raster_to_vector(georeferenced_image_path, vector_shape_file_path, threshold_value=240):
    with rasterio.open(georeferenced_image_path) as src:
        raster = src.read(1)  # Read the first band
        transform = src.transform
        raster_crs = src.crs  # Get the CRS of the raster

    threshold_value = 128
    binary_raster = (raster > threshold_value).astype(np.uint8)

    shapes_generator = shapes(binary_raster, transform=transform)
    polygons = []
    for geom, value in shapes_generator:
        if value == 1:
            polygons.append(shape(geom))

    output_path = vector_shape_file_path

    crs_epsg_code = 4326

    schema = {
        'geometry': 'Polygon',
        'properties': {'id': 'int'},
    }

    crs = from_epsg(crs_epsg_code)  
    with fiona.open(output_path, 'w', driver='ESRI Shapefile', schema=schema, crs=crs) as shp:
        for i, polygon in enumerate(polygons):
            shp.write({
               'geometry': mapping(polygon),
                'properties': {'id': i},
            })

    print(f"Vector file saved at: {vector_shape_file_path}")

In [29]:
transform, crs = get_georeferencing_info(main_tif_path)
geo_reference_jpg(merged_predicted_image_path, georeferenced_image_path, transform, crs)
raster_to_vector(georeferenced_image_path, vector_shape_file_path, threshold_value=240)

Vector file saved at: /home/jazzy/sem-farm/Spatial_temporal/shapefile/22spa.shp


/tmp/ipykernel_4019/1627528846.py:3: FionaDeprecationWarning: This function will be removed in version 2.0. Please use CRS.from_epsg() instead.
  raster_to_vector(georeferenced_image_path, vector_shape_file_path, threshold_value=240)
